In [3]:
import os
import logging
from dotenv import load_dotenv
from pathlib import Path
import sys

load_dotenv()
logging.basicConfig(level=logging.INFO)

# Use current working directory instead of __file__
PROJECT_ROOT = Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from model_providers import LLMFactory

factory = LLMFactory()
print("Available providers:", factory.available_providers)


Available providers: ['openai', 'anthropic', 'gemini', 'groq', 'ollama', 'nvidia', 'cerebras', 'nebius', 'openrouter', 'huggingface', 'vllm']


In [4]:
# Swap provider name to switch the entire agent brain
llm = factory.get_llm("nebius", model_name="MiniMaxAI/MiniMax-M2.5", temperature=0.5)

# Quick sanity check
response = llm.invoke("Hello, are you ready?")
print(response.content)

INFO:model_providers:[Nebius] Initializing model: MiniMaxAI/MiniMax-M2.5
INFO:model_providers:[Nebius] Model 'MiniMaxAI/MiniMax-M2.5' initialized successfully.
INFO:httpx:HTTP Request: POST https://api.tokenfactory.nebius.com/v1/chat/completions "HTTP/1.1 200 OK"


<think>The user is asking if I'm ready. I should respond in a friendly, helpful manner and indicate that I'm ready to assist them.
</think>

Hello! Yes, I'm ready to help. How can I assist you today?


In [5]:
from qdrant_client import QdrantClient
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings
from langchain_qdrant import QdrantVectorStore, FastEmbedSparse, RetrievalMode

QDRANT_URL          = os.getenv("QDRANT_CLUSTER_URL", "")
QDRANT_API_KEY      = os.getenv("QDRANT_API_KEY", "")
COLLECTION_NAME     = "multimodal-rag"
IMAGE_COLLECTION_NAME = "multimodal-rag-images"

client            = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)
embeddings        = NVIDIAEmbeddings(model="baai/bge-m3")
sparse_embeddings = FastEmbedSparse(model_name="Qdrant/bm25")

vectorstore = QdrantVectorStore(
    client=client,
    collection_name=COLLECTION_NAME,
    embedding=embeddings,
    sparse_embedding=sparse_embeddings,
    retrieval_mode=RetrievalMode.HYBRID,
    vector_name="dense",
    sparse_vector_name="sparse",
)

image_vectorstore = QdrantVectorStore(
    client=client,
    collection_name=IMAGE_COLLECTION_NAME,
    embedding=embeddings,
    sparse_embedding=sparse_embeddings,
    retrieval_mode=RetrievalMode.HYBRID,
    vector_name="dense",
    sparse_vector_name="sparse",
)

print("Vectorstore connected.")
print(f"Points: {client.get_collection(COLLECTION_NAME).points_count}")

INFO:httpx:HTTP Request: GET https://0b25926a-e09f-4677-8e3f-f54223294158.sa-east-1-0.aws.cloud.qdrant.io:6333/collections/multimodal-rag "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://0b25926a-e09f-4677-8e3f-f54223294158.sa-east-1-0.aws.cloud.qdrant.io:6333 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://0b25926a-e09f-4677-8e3f-f54223294158.sa-east-1-0.aws.cloud.qdrant.io:6333/collections/multimodal-rag "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://0b25926a-e09f-4677-8e3f-f54223294158.sa-east-1-0.aws.cloud.qdrant.io:6333/collections/multimodal-rag-images "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://0b25926a-e09f-4677-8e3f-f54223294158.sa-east-1-0.aws.cloud.qdrant.io:6333/collections/multimodal-rag-images "HTTP/1.1 200 OK"


Vectorstore connected.


INFO:httpx:HTTP Request: GET https://0b25926a-e09f-4677-8e3f-f54223294158.sa-east-1-0.aws.cloud.qdrant.io:6333/collections/multimodal-rag "HTTP/1.1 200 OK"


Points: 227


In [ ]:
from langchain_core.tools import tool
from langchain_core.documents import Document

@tool
def retrieve_text_context(query: str) -> str:
    """
    Search the knowledge base for relevant text passages.
    Use this tool when the user asks a factual question about the document.
    Returns the top 4 most relevant passages with page and type metadata.
    """
    results = vectorstore.similarity_search_with_score(query, k=4)

    if not results:
        return "No relevant text found in the knowledge base."

    formatted = []
    for doc, score in results:
        page  = doc.metadata.get("page", "?")
        dtype = doc.metadata.get("type", "?")
        formatted.append(
            f"[Page {page} | {dtype} | score={score:.4f}]\n{doc.page_content}"
        )

    return "\n\n---\n\n".join(formatted)


@tool
def retrieve_image_context(query: str) -> str:
    """
    Search for relevant images in the knowledge base.
    Use this tool when the user asks about diagrams, figures, or visual content.
    Returns image metadata and OCR text extracted from figures.
    """
    img_count = client.get_collection(IMAGE_COLLECTION_NAME).points_count or 10
    retriever = image_vectorstore.as_retriever(search_kwargs={"k": img_count})
    image_docs = retriever.invoke(query)

    if not image_docs:
        return "No relevant images found."

    formatted = []
    for doc in image_docs:
        page       = doc.metadata.get("page", "?")
        image_path = doc.metadata.get("image_path", "not available")
        ocr_text   = doc.page_content[:200]
        formatted.append(
            f"[Image | Page {page}]\n  path : {image_path}\n  ocr  : {ocr_text}"
        )

    return "\n\n".join(formatted)


@tool
def search_by_element_type(element_type: str, query: str) -> str:
    """
    Search the knowledge base filtered by a specific element type.
    Valid element types: Title, NarrativeText, Table, FigureCaption, ListItem, Formula.
    Use this when the user specifically asks for tables, titles, formulas, or figure captions.
    """
    from qdrant_client.http.models import Filter, FieldCondition, MatchValue

    results = vectorstore.similarity_search_with_score(
        query,
        k=4,
        filter=Filter(
            must=[
                FieldCondition(
                    key="metadata.type",
                    match=MatchValue(value=element_type)
                )
            ]
        )
    )

    if not results:
        return f"No elements of type '{element_type}' found for query: {query}"

    formatted = []
    for doc, score in results:
        page = doc.metadata.get("page", "?")
        formatted.append(
            f"[Page {page} | {element_type} | score={score:.4f}]\n{doc.page_content}"
        )

    return "\n\n---\n\n".join(formatted)


tools = [retrieve_text_context, retrieve_image_context, search_by_element_type]

print("Tools defined:")
for t in tools:
    print(f"  {t.name}: {t.description[:80]}")